In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import split, lit
from pyspark.sql.functions import col,to_timestamp,when

spark = SparkSession.builder \
    .appName("Jupyter").config("spark.driver.memory", "6g").config("spark.sql.shuffle.partitions", "100").getOrCreate()

# Disabling automatic broadcast join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# Explicitly broadcast JOINs medals and maps
# I stopped the broadcast JOINs medals and maps because they have nothing to do in common so I guess this one is just and instruction to use those two tables as broadcast because they aren't heavy data

# medals = spark.read.option("header", "true").csv('/home/iceberg/data/medals.csv')
# maps = spark.read.option("header", "true").csv('/home/iceberg/data/maps.csv')

#broadcast_joined_df = medals.join(broadcast(maps))
# broadcast_joined_df.show()

# Bucket join match_details, matches, and medal_matches_players on match_id with 16 buckets

bucketed_match_details = spark.read.option("header","true").csv("/home/iceberg/data/match_details.csv")
bucketed_medal_matches_players = spark.read.option("header","true").csv("/home/iceberg/data/medals_matches_players.csv")
bucketed_matches = spark.read.option("header","true").csv("/home/iceberg/data/matches_copy.csv")

spark.sql("DROP TABLE IF EXISTS bootcamp.matches_bucketed")
spark.sql("DROP TABLE IF EXISTS bootcamp.match_details_bucketed")
spark.sql("DROP TABLE IF EXISTS bootcamp.medals_matches_players_bucketed")


bucketedMatchesDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
     match_id STRING,
     mapid STRING,
     is_team_game BOOLEAN,
     playlist_id STRING,
     completion_date TIMESTAMP
 )
 USING iceberg
 PARTITIONED BY (completion_date, bucket(16, match_id))
"""
spark.sql(bucketedMatchesDDL)

bucketedDetailsDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.match_details_bucketed (
    match_id STRING,
    player_gamertag STRING,
    player_total_kills INT,
    player_total_deaths INT
)
USING iceberg
PARTITIONED BY (bucket(16, match_id))
"""
spark.sql(bucketedDetailsDDL)

bucketedMedalsDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.medals_matches_players_bucketed (
    match_id STRING,
    player_gamertag STRING,
    medal_id STRING,
    count INT
)
USING iceberg
PARTITIONED BY (bucket(16, match_id))
"""
spark.sql(bucketedMedalsDDL)




bucketed_medal_matches_players.select(
    "match_id", "player_gamertag", "medal_id", col("count").cast("integer")
).write \
  .mode("append") \
  .bucketBy(16, "match_id") \
  .saveAsTable("bootcamp.medals_matches_players_bucketed")

bucketed_match_details.select(
    "match_id", "player_gamertag", col("player_total_kills").cast("integer"), col("player_total_deaths").cast("integer")
).write \
  .mode("append") \
  .bucketBy(16, "match_id") \
  .saveAsTable("bootcamp.match_details_bucketed")

bucketed_matches_transformed = bucketed_matches.select(
    "match_id", "mapid", col("is_team_game").cast("boolean"), "playlist_id",
    to_timestamp("completion_date", "yyyy-MM-dd HH:mm:ss.SSSSSS").alias("completion_date")
)
# Try a balanced number; usually >= number of cores
#bucketed_matches_transformed = bucketed_matches_transformed.repartition(200)

# Explicit broadcast join example
maps = spark.read.option("header", "true").csv("/home/iceberg/data/maps.csv")
matches = spark.table("bootcamp.matches_bucketed")
matches_with_map_details = matches.join(broadcast(maps), on="mapid")
matches_with_map_details.select("match_id", "map_name", "playlist_id").show(5)



# Which player averages the most kills per game?
most_kills_per_game = matches_details.groupBy("player_gamertag") \
    .agg(
        (sum("player_total_kills") / countDistinct("match_id")).alias("avg_kills_per_game")
    ).orderBy(col("avg_kills_per_game").desc())


# Which playlist gets played the most?

most_played_playlists = matches.groupBy("playlist_id") \
    .agg(count("*").alias("times_played")) \
    .orderBy(col("times_played").desc())


# Which map gets played the most?
most_played_maps = matches.groupBy("mapid") \
    .agg(count("*").alias("times_played")) \
    .orderBy(col("times_played").desc())

# Which map do players get the most Killing Spree medals on?
medals = spark.read.option("header", "true").csv("/home/iceberg/data/medals.csv")
medals_broadcasted = broadcast(medals)

medal_matches_named = medal_matches.join(medals_broadcasted, medal_matches.medal_id == medals.medal_id)

killing_spree_medals = medal_matches_named.filter(col("medal_name") == "Killing Spree")

killing_spree_with_map = killing_spree_medals.join(matches, on="match_id")

most_killing_spree_by_map = killing_spree_with_map.groupBy("mapid") \
    .agg(sum("count").alias("total_killing_sprees")) \
    .orderBy(col("total_killing_sprees").desc())
# Sort within partitions
matches_sorted = matches.sortWithinPartitions("playlist_id")

matches = spark.table("bootcamp.matches_bucketed")
matches_details = spark.table("bootcamp.match_details_bucketed")
medal_matches = spark.table("bootcamp.medals_matches_players_bucketed")

joined_df = matches_details.join(medal_matches,on='match_id')
joined_df.show()

spark.stop()



25/07/09 18:17:48 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/07/09 18:17:56 ERROR Utils: Aborting task                        (0 + 1) / 1]
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.io.ByteArrayOutputStream.<init>(ByteArrayOutputStream.java:81)
	at org.apache.iceberg.shaded.org.apache.parquet.hadoop.CodecFactory$HeapBytesCompressor.<init>(CodecFactory.java:222)
	at org.apache.iceberg.shaded.org.apache.parquet.hadoop.CodecFactory.createCompressor(CodecFactory.java:273)
	at org.apache.iceberg.shaded.org.apache.parquet.hadoop.CodecFactory.getCompressor(CodecFactory.java:255)
	at org.apache.iceberg.parquet.ParquetWriter.<init>(ParquetWriter.java:90)
	at org.apache.iceberg.parquet.Parquet$WriteBuilder.build(Parquet.java:393)
	at org.apache.iceberg.parquet.Parquet$DataWriteBuilder.build(Parquet.java:787)
	at org.apache.iceberg.data.BaseFileWriterFactory.newDataWriter(BaseFileWriterFactory.java:131)
	at org.apac

Py4JError: py4j does not exist in the JVM

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [ ]:
bucketed_matches_transformed \
  .write \
  .mode("append") \
  .partitionBy("completion_date") \
  .bucketBy(16, "match_id") \
  .saveAsTable("bootcamp.matches_bucketed")
bucketed_matches_transformed.show()

In [7]:
%%sql
DROP TABLE bootcamp.match_details_bucketed

Py4JJavaError: An error occurred while calling o145.sql.
: org.apache.iceberg.exceptions.ServiceFailureException: Server error: NotFoundException: Location does not exist: s3://warehouse/bootcamp/match_details_bucketed/metadata/00000-3d0a8c55-646d-4913-8b50-fac7c05d5221.metadata.json
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:217)
	at org.apache.iceberg.rest.ErrorHandlers$TableErrorHandler.accept(ErrorHandlers.java:118)
	at org.apache.iceberg.rest.ErrorHandlers$TableErrorHandler.accept(ErrorHandlers.java:102)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:224)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:308)
	at org.apache.iceberg.rest.BaseHTTPClient.get(BaseHTTPClient.java:77)
	at org.apache.iceberg.rest.RESTClient.get(RESTClient.java:97)
	at org.apache.iceberg.rest.RESTSessionCatalog.loadInternal(RESTSessionCatalog.java:465)
	at org.apache.iceberg.rest.RESTSessionCatalog.loadTable(RESTSessionCatalog.

In [5]:
spark.stop()


ConnectionRefusedError: [Errno 111] Connection refused